# MPIA Arxiv on Deck 2

Contains the steps to produce the paper extractions.

In [1]:
# Imports
import os
from IPython.display import Markdown, display
from tqdm.notebook import tqdm
import warnings
from PIL import Image 
import re

# requires arxiv_on_deck_2

from arxiv_on_deck_2.arxiv2 import (get_new_papers, 
                                    get_paper_from_identifier,
                                    retrieve_document_source, 
                                    get_markdown_badge)
from arxiv_on_deck_2 import (latex,
                             latex_bib,
                             mpia,
                             highlight_authors_in_list)

# Sometimes images are really big
Image.MAX_IMAGE_PIXELS = 1000000000 

In [2]:
# Some useful definitions.

class AffiliationWarning(UserWarning):
    pass

class AffiliationError(RuntimeError):
    pass

def validation(source: str):
    """Raises error paper during parsing of source file
    
    Allows checks before parsing TeX code.
    
    Raises AffiliationWarning
    """
    check = mpia.affiliation_verifications(source, verbose=True)
    if check is not True:
        raise AffiliationError("mpia.affiliation_verifications: " + check)

        
warnings.simplefilter('always', AffiliationWarning)


def get_markdown_qrcode(paper_id: str):
    """ Generate a qrcode to the arxiv page using qrserver.com
    
    :param paper: Arxiv paper
    :returns: markdown text
    """
    url = r"https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="
    txt = f"""<img src={url}"https://arxiv.org/abs/{paper_id}">"""
    txt = '<div id="qrcode">' + txt + '</div>'
    return txt


def clean_non_western_encoded_characters_commands(text: str) -> str:
    """ Remove non-western encoded characters from a string
    List may need to grow.
    
    :param text: the text to clean
    :return: the cleaned text
    """
    text = re.sub(r"(\\begin{CJK}{UTF8}{gbsn})(.*?)(\\end{CJK})", r"\2", text)
    return text


def get_initials(name: str) -> str:
    """ Get the short name, e.g., A.-B. FamName
    :param name: full name
    :returns: initials
    """
    initials = []
    # account for non western names often in ()
    if '(' in name:
        name = clean_non_western_encoded_characters_commands(name)
        suffix = re.findall(r"\((.*?)\)", name)[0]
        name = name.replace(f"({suffix})", '')
    else:
        suffix = ''
    split = name.split()
    for token in split[:-1]:
        if '-' in token:
            current = '-'.join([k[0] + '.' for k in token.split('-')])
        else:
            current = token[0] + '.'
        initials.append(current)
    initials.append(split[-1].strip())
    if suffix:
        initials.append(f"({suffix})")
    return ' '.join(initials)

## get list of arxiv paper candidates

We use the MPIA mitarbeiter list webpage from mpia.de to get author names
We then get all new papers from Arxiv and match authors

In [3]:
# deal with the author list and edge cases of people that cannot be consistent on their name  

def filter_non_scientists(name: str) -> bool:
    """ Loose filter on expected authorships

    removing IT, administration, technical staff
    :param name: name
    :returns: False if name is not a scientist
    """
    remove_list = ['Licht', 'Binroth', 'Witzel', 'Jordan',
                   'Zähringer', 'Scheerer', 'Hoffmann', 'Düe',
                   'Hellmich', 'Enkler-Scharpegge', 'Witte-Nguy',
                   'Dehen', 'Beckmann', 'Jager', 'Jäger'
                  ]

    for k in remove_list:
        if k in name:
            return False
    return True

def add_author_to_list(author_list: list) -> list:
    """ Add author to list if not already in list
    
    :param author: author name
    :param author_list: list of authors
    :returns: updated list of authors
    """
    add_list = ['T. Henning', 'R. E. Hviding']

    for author in add_list:
        if author not in author_list:
            author_list.append(author)
    return author_list

# get list from MPIA website
# filter for non-scientists (mpia.get_mpia_mitarbeiter_list() does some filtering)
mpia_authors = [k[1] for k in mpia.get_mpia_mitarbeiter_list() if filter_non_scientists(k[1])]
# add some missing author because of inconsistencies in their MPIA name and author name on papers
mpia_authors = add_author_to_list(mpia_authors)

In [4]:
new_papers = get_new_papers()
# add manual references
add_paper_refs = []
new_papers.extend([get_paper_from_identifier(k) for k in add_paper_refs])

def robust_call(fn, value, *args, **kwargs):
    try:
        return fn(value, *args, **kwargs)
    except Exception:
        return value

candidates = []
for paperk in new_papers:
    # Check author list with their initials
    normed_author_list = [robust_call(mpia.get_initials, k) for k in paperk['authors']]
    hl_authors = highlight_authors_in_list(normed_author_list, mpia_authors, verbose=True)
    matches = [(hl, orig) for hl, orig in zip(hl_authors, paperk['authors']) if 'mark' in hl]
    paperk['authors'] = hl_authors
    if matches:
        # only select paper if an author matched our list
        candidates.append(paperk)
print("""Arxiv has {0:,d} new papers today""".format(len(new_papers)))        
print("""          {0:,d} with possible author matches""".format(len(candidates)))

H. Zhang  ->  H. Zhang  |  ['H. Zhang']
G. Chauvin  ->  G. Chauvin  |  ['G. Chauvin']
T. Henning  ->  T. Henning  |  ['T. Henning']
L. Kreidberg  ->  L. Kreidberg  |  ['L. Kreidberg']
M. Samland  ->  M. Samland  |  ['M. Samland']
A. Kahle  ->  K. A. Kahle  |  ['A. Kahle']
L. Kreidberg  ->  L. Kreidberg  |  ['L. Kreidberg']
J. Li  ->  J. Li  |  ['J. Li']
M. Zhang  ->  M. Zhang  |  ['M. Zhang']
B. Maity  ->  B. Maity  |  ['B. Maity']
Arxiv has 90 new papers today
          6 with possible author matches


# Parse sources and generate relevant outputs

From the candidates, we do the following steps:
* get their tarball from ArXiv (and extract data)
* find the main .tex file: find one with \documentclass{...} (sometimes it's non trivial)
* Check affiliations with :func:`validation`, which uses :func:`mpia.affiliation_verifications`
* If passing the affiliations: we parse the .tex source
   * inject sub-documents into the main (flatten the main document)
   * parse structure, extract information (title, abstract, authors, figures...)
   * handles `\graphicspath` if provided
* Generate the .md document.

In [5]:
documents = []
failed = []
for paper in tqdm(candidates):
    # debug crap
    paper['identifier'] = paper['identifier'].lower().replace('arxiv:', '').replace(r'\n', '').strip()
    paper_id = paper['identifier']
    
    folder = f'tmp_{paper_id}'

    try:
        if not os.path.isdir(folder):
            folder = retrieve_document_source(f"{paper_id}", f'tmp_{paper_id}')
        
        try:
            doc = latex.LatexDocument(folder, validation=validation)    
        except AffiliationError as affilerror:
            msg = f"ArXiv:{paper_id:s} is not an MPIA paper... " + str(affilerror)
            failed.append((paper, "affiliation error: " + str(affilerror) ))
            continue
        
        # Hack because sometimes author parsing does not work well
        if (len(doc.authors) != len(paper['authors'])):
            doc._authors = paper['authors']
        else:
            # highlight authors (FIXME: doc.highlight_authors)
            # done on arxiv paper already
            doc._authors = highlight_authors_in_list(
                [get_initials(k) for k in doc.authors], 
                mpia_authors, verbose=True)
        if (doc.abstract) in (None, ''):
            doc._abstract = paper['abstract']
            
        doc.comment = (get_markdown_badge(paper_id) + 
                       "<mark>Appeared on: " + paper['date'] + "</mark> - ")
        if paper['comments']:
            doc.comment += " _" + paper['comments'] + "_"
        
        full_md = doc.generate_markdown_text()
        
        full_md += get_markdown_qrcode(paper_id)
        
        # replace citations
        try:
            bibdata = latex_bib.LatexBib.from_doc(doc)
            full_md = latex_bib.replace_citations(full_md, bibdata)
        except Exception as e:
            print("Issues with the citations")
            print(e)
        
        documents.append((paper_id, full_md))
    except Exception as e:
        warnings.warn(latex.LatexWarning(f"{paper_id:s} did not run properly\n" +
                                         str(e)
                                        ))
        failed.append((paper, "latex error " + str(e)))

  0%|          | 0/6 [00:00<?, ?it/s]

Retrieving document from  https://arxiv.org/e-print/2611.25039
extracting tarball to tmp_2611.25039...

 done.


Retrieving document from  https://arxiv.org/e-print/2611.25120
extracting tarball to tmp_2611.25120...

 done.


G. Chauvin  ->  G. Chauvin  |  ['G. Chauvin']
T. Henning  ->  T. Henning  |  ['T. Henning']
L. Kreidberg  ->  L. Kreidberg  |  ['L. Kreidberg']
M. Samland  ->  M. Samland  |  ['M. Samland']


... (2 outputs left out)

A. Kahle  ->  K. A. Kahle  |  ['A. Kahle']
L. Kreidberg  ->  L. Kreidberg  |  ['L. Kreidberg']


Retrieving document from  https://arxiv.org/e-print/2611.25348
extracting tarball to tmp_2611.25348... done.
Retrieving document from  https://arxiv.org/e-print/2611.25384
extracting tarball to tmp_2611.25384... done.
Retrieving document from  https://arxiv.org/e-print/2611.25823
extracting tarball to tmp_2611.25823...

 done.


/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3823: LatexWarning: Multiple tex files.

  exec(code_obj, self.user_global_ns, self.user_ns)
/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3823: LatexWarning: Found documentclass in tmp_2611.25823/main.tex

  exec(code_obj, self.user_global_ns, self.user_ns)
/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/arxiv_on_deck_2/latex.py:414: LatexWarning: Latex injecting: 'Institutes_latex' from 'tmp_2611.25823/Institutes_latex.tex'
  warnings.warn(LatexWarning(f"Latex injecting: '{ext}' from '{subsource}'"))
/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/arxiv_on_deck_2/latex.py:414: LatexWarning: Latex injecting: 'team_acknowledgements' from 'tmp_2611.25823/team_acknowledgements.tex'
  warnings.warn(LatexWarning(f"Latex injecting: '{ext}' from '{subsource}'"))
/opt/hostedtoolcache/Python/3.11.

### Export the logs

Throughout, we also keep track of the logs per paper. see `logs-{today date}.md` 

In [6]:
import datetime
today = str(datetime.date.today())
logfile = f"_build/html/logs/log-{today}.md"


with open(logfile, 'w') as logs:
    # Success
    logs.write(f'# Arxiv on Deck 2: Logs - {today}\n\n')
    logs.write("""* Arxiv had {0:,d} new papers\n""".format(len(new_papers)))
    logs.write("""    * {0:,d} with possible author matches\n\n""".format(len(candidates)))
    logs.write("## Sucessful papers\n\n")
    display(Markdown("## Successful papers"))
    success = [k[0] for k in documents]
    for candid in candidates:
        if candid['identifier'].split(':')[-1] in success:
            display(candid)
            logs.write(candid.generate_markdown_text() + '\n\n')

    ## failed
    logs.write("## Failed papers\n\n")
    display(Markdown("## Failed papers"))
    failed = sorted(failed, key=lambda x: x[1])
    current_reason = ""
    for paper, reason in failed:
        if 'affiliation' in reason:
            color = 'green'
        else:
            color = 'red'
        data = Markdown(
                paper.generate_markdown_text() + 
                f'\n|<p style="color:{color:s}"> **ERROR** </p>| <p style="color:{color:s}">{reason:s}</p> |'
               )
        if reason != current_reason:
            logs.write(f'### {reason:s} \n\n')
            current_reason = reason
        logs.write(data.data + '\n\n')
        
        # only display here the important errors (all in logs)
        # if color in ('red',):
        display(data)

## Successful papers


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.25120-b31b1b.svg)](https://arxiv.org/abs/2611.25120) | **On the High Eccentricity Orbit of 51 Eri b and Constraints on Interactions with a Second Companion**  |
|| W. O. Balmer, et al. -- incl., <mark>G. Chauvin</mark>, <mark>T. Henning</mark>, <mark>L. Kreidberg</mark>, <mark>M. Samland</mark> |
|*Appeared on*| *2026-11-20*|
|*Comments*| *Re-submitted to the Astronomical Journal*|
|**Abstract**|            The eccentricity of an exoplanet's orbit is a key observational quantity that can be used to infer its dynamical past and predict its future orbital evolution. Planets are expected to emerge from their protoplanetary cradles with roughly circular eccentricities, and non-circular eccentricites to arise from subsequent dynamical processing in the absence of the gas disk. We present an updated orbit for 51 Eridani b based on 5 years of 0.2 mas precision optical interferometry from VLTI/GRAVITY(+). This planet is young (${\sim}20\,\mathrm{Myr}$ old), temperate ($T_{\rm eff}\sim650\,\mathrm{K}$), and on a wide orbit ($a{=}9.36\,\mathrm{au}$) with an uncertain mass ($M_{\rm b}\leq12\,M_{\rm J}$). We measured the planet's motion to an exceptionally high precision, confirming that its orbit is very eccentric ($e_b=0.574\pm0.003$). We observe residuals to the absolute astrometry of the host star based on a single planet orbit that could be attributable to reflex motion from a third body in the system. We construct a two companion orbit model that could explain the apparent reflex motion of 51 Eri A: this model predicts that the putative second companion has $M_c\in[1.0, 2900]_{95\%}\,M_{\rm Jup}$ and $a_c\in[0.3, 3.1]_{95\%}\,\mathrm{au}$. In this framework, a dynamical interaction between b and ``c" could have kicked the exterior, lower mass b out onto its present, eccentric orbit. If this signal is astrophysical, and not due to a systematic error, this candidate should be confirmable with Gaia DR4 epoch astrometry in the near future.         |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.25230-b31b1b.svg)](https://arxiv.org/abs/2611.25230) | **The SPACE Program IV: Atmospheric Features on the Cold Transiting Sub-Neptune TOI-6478 b**  |
|| A. Triantafillides, et al. -- incl., <mark>A. Kahle</mark>, <mark>L. Kreidberg</mark> |
|*Appeared on*| *2026-11-20*|
|*Comments*| *16 pages, 9 Figure, 4 Tables, Submitted to AJ*|
|**Abstract**|            Among the current population of exoplanets there are very few sub-Neptunes that have their atmospheric composition identified. This is primarily due to sub-Neptunes having high metallicity, clouds, and/or hazes obscuring absorption features. This often produces a spectrum that presents itself as a featureless spectrum especially in the near-IR. However, there are tentative trends that suggest sub-Neptunes with the lowest equilibrium temperatures ($\mathrm{T_{eq} < 400\:K}$) have detectable features, perhaps due to inefficient haze production or cloud sedimentation. TOI-6478 b is a low-density sub-Neptune exoplanet ($\rm <\mathrm{9.9\:M_{\oplus}}$ and $\rm \sim 4.5R_{\oplus}$) with an equilibrium temperature $\mathrm{T_{eq} = 220 \:K}$ assuming zero albedo. In this study, we present the highest S/N spectral feature ($\sim$6) of any sub-Neptune detected using HST/WFC3. We detect this spectral feature using a robust analysis that incorporates the Transit Light Source Effect (TLSE) to rule out false detections due to stellar activity. With the narrow bandpass of HST we were unable to distinguish whether the spectral feature at 1.4$\mu$m is from H$_2$O or CH$_4$ absorption. Meanwhile, self-consistent atmospheric models provide physical motivation for CH$_4$ to be the dominant absorber given that H$_2$O should condense or cold trap at higher pressures. Through our analysis we determine that HST/WFC3 alone cannot distinguish between major absorbers. However, a broader wavelength coverage by JWST will be able to measure the atmospheric composition in detail, due to the high S/N features for this target.         |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.25823-b31b1b.svg)](https://arxiv.org/abs/2611.25823) | **Square Kilometre Array Science Data Challenge 3b: EoR inference**  |
|| O. Bait, et al. -- incl., <mark>B. Maity</mark> |
|*Appeared on*| *2026-11-20*|
|*Comments*| *32 pages, 13 figures, submitted to MNRAS*|
|**Abstract**|            We present and analyse the results of Science Data Challenge 3b (SDC3b; this https URL), an Epoch of Reionization (EoR) inference exercise organised by the Square Kilometre Array Observatory (SKAO) using simulated SKA data. The nine-month challenge, running from October 2024 to May 2025, represents the first community-wide comparison of EoR inference pipelines on common simulated datasets. We consider multiple datasets designed to assess performance on both foreground-free and foreground-contaminated cylindrical power spectra and imaging products, adopting different reionization scenarios and EoR simulation models. We include results from 16 teams, which employ a range of methodologies to infer the posterior distribution of the hydrogen neutral fraction across three redshift intervals spanning z=8.41 to z=6.25. The results show significant inter-team variability of the astrophysical constraints even in the simpler case where only instrumental noise is added to the cosmological signal. This underscores the need for further investigation into the relative contributions of signal modelling, the treatment of instrumental effects, and inference methods to the robustness of the resulting conclusions. Although individual methods can produce biased parameter estimates, the combined average of their estimates remains close to the input values, suggesting that independent approaches can offset method-specific biases.         |

## Failed papers


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.25039-b31b1b.svg)](https://arxiv.org/abs/2611.25039) | **Twisted resonant orbits: Quantifying the deceleration of the Galactic bar**  |
|| A. Archer, A. Dillamore, <mark>H. Zhang</mark>, V. Belokurov |
|*Appeared on*| *2026-11-20*|
|*Comments*| *18 pages, 23 Figures*|
|**Abstract**|            Recent structural discoveries in Gaia Data Release 3 reveal distinct, chevron-shaped over-densities in the radial phase space of the halo. Although initial interpretations identify these features as the phase-mixed relics of a past galaxy merger, their prominent asymmetry and persistence at high metallicities strongly challenge this interpretation. These features are better explained as a dynamical response to a Galactic bar, where stars are trapped in resonances. We establish the fractional $v_r$ asymmetry of the chevrons as a quantitative diagnostic tool for the bar's rotational state. We utilise test-particle simulations of a Milky Way-like galaxy to model various bar pattern speed evolutions, including decelerating and constant speed scenarios. By mapping the fractional asymmetry of the phase-space chevrons across a parameter space of bar deceleration rates and pattern speeds, we establish a direct correlation between the bar's deceleration and the rotational response of the resonant orbits. Comparing these simulated constraints with observational data from Gaia DR3 provides a framework for measuring the current deceleration of the Galactic bar, and a probe of the amount of dynamical friction occurring between the baryonic and dark matter content of the Milky Way. We find a dimensionless deceleration parameter $\eta \equiv - \dot{\Omega}_b/\Omega_b^2 = 0.0015^{+0.0016}_{-0.0009}$, equivalent to $\dot{\Omega}_b = -2.23^{+1.50}_{-2.46} \, \text{km s}^{-1} \text{kpc}^{-1} \text{Gyr}^{-1}$, in agreement with previous estimates obtained with independent methods. Gaia DR4 should sharpen this measurement considerably.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.25348-b31b1b.svg)](https://arxiv.org/abs/2611.25348) | **FAST Burst-Cluster Catalog of FRB 20240114A**  |
|| L.-X. Zhang, et al. -- incl., <mark>J. Li</mark> |
|*Appeared on*| *2026-11-20*|
|*Comments*| *14 pages, 9 figures, under review at ApJS*|
|**Abstract**|            We present the complete FAST burst-cluster catalog of FRB 20240114A, covering its 495-day active episode from January 28, 2024 to June 5, 2025. The catalog contains 13261 burst clusters, 19037 bursts, and 29026 sub-bursts detected in 99 observing sessions. The data reveal distinct organization across temporal scales. Among drifting burst clusters, downward-drifting clusters outnumber upward-drifting ones by approximately $9.4:1$. Between separate burst clusters, previously reported frequency changes show no preferred direction, while our Weibull analysis finds their arrival times broadly consistent with Poisson statistics, suggesting that burst clusters may represent approximately independent emission episodes within observing sessions. The Parkes/Murriyang observations revealed a long-term modulation of the characteristic emission frequency, and the extended FAST measurements show a consistent evolution. Burst energetics increase with cluster multiplicity and activity level, although significant scatter remains. The manually refined dispersion measure increases at $1.29\pm0.11\,\mathrm{pc\,cm^{-3}\,yr^{-1}}$, enlarging the still limited but growing sample of repeaters with increasing DM. Overall, these results summarize the spectral--temporal evolution of FRB 20240114A and present a burst-cluster catalog linked to the existing BurstID measurements, providing a basis for future studies of repeating FRBs.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.25384-b31b1b.svg)](https://arxiv.org/abs/2611.25384) | **A Shallow Phase-resolved Gamma-Ray Luminosity-Cutoff Energy Relation in Globular Cluster Millisecond Pulsars**  |
|| <mark>M. Zhang</mark>, J. Zhang, S. Chang, P. Zhang |
|*Appeared on*| *2026-11-20*|
|*Comments*| *Submitted to ApJ, 14 pages, 4 figures*|
|**Abstract**|            Globular clusters (GCs) provide dense stellar environments for investigating the high-energy emission of millisecond pulsars (MSPs). Using $\sim$17.3~yr of \textit{Fermi}-LAT Pass~8 data in the 0.1-500.0~GeV range, we study four MSPs that dominate, or contribute substantially to, the GeV $\gamma$-ray emission of their host GCs. Significant off-pulse $\gamma$-ray emission is detected from PSR~J1823$-$3021A, J1824$-$2452A, and J1835$-$3259B. For these sources, the cutoff energy, $E_{\rm cut}$, varies systematically with rotational phase and is positively correlated with the phase-resolved energy flux. Using the phase-resolved isotropic $\gamma$-ray luminosity ($L_{\gamma}$), we find $L_{\gamma}\propto E_{\rm cut}^{\alpha}$ with $\alpha=0.74^{+0.16}_{-0.14}$. This slope is substantially shallower than that reported for field MSPs, $\alpha=2.31^{+0.22}_{-0.25}$, and the $\alpha=16/7\simeq2.29$ scaling expected for curvature radiation in the equatorial current sheet framework. The origin of the shallow relation remains uncertain, potentially reflecting cluster-associated $\gamma$-ray emission, differences in emission geometry and beaming, or limited photon statistics. Our results demonstrate that GC MSPs provide valuable targets for testing phase-resolved high-energy emission physics in dense stellar environments.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |

## Export documents

We now write the .md files and export relevant images

In [7]:
def export_markdown_summary(md: str, md_fname:str, directory: str):
    """Export MD document and associated relevant images"""
    import os
    import shutil
    import re

    if (os.path.exists(directory) and not os.path.isdir(directory)):
        raise RuntimeError(f"a non-directory file exists with name {directory:s}")

    if (not os.path.exists(directory)):
        print(f"creating directory {directory:s}")
        os.mkdir(directory)

    fig_fnames = (re.compile(r'\[Fig.*\]\((.*)\)').findall(md) + 
                  re.compile(r'\<img src="([^>\s]*)"[^>]*/>').findall(md))
    print("found figures", fig_fnames)
    for fname in fig_fnames:
        if 'http' in fname:
            # No need to copy online figures
            continue
        if not os.path.exists(fname):
            print("file not found", fname)
            continue
        print("copying ", fname, "to", directory)
        destdir = os.path.join(directory, os.path.dirname(fname))
        destfname = os.path.join(destdir, os.path.basename(fname))
        try:
            os.makedirs(destdir)
        except FileExistsError:
            pass
        shutil.copy(fname, destfname)
    with open(os.path.join(directory, md_fname), 'w') as fout:
        fout.write(md)
    print("exported in ", os.path.join(directory, md_fname))
    [print("    + " + os.path.join(directory,fk)) for fk in fig_fnames]

In [8]:
for paper_id, md in documents:
    export_markdown_summary(md, f"{paper_id:s}.md", '_build/html/')

found figures ['tmp_2611.25120/./cEri_2021-10-28-irdis_fmmf_all_gravi_pointings.png', 'tmp_2611.25120/./cEri_2022-01-23_detection_chi2.png', 'tmp_2611.25120/./cEri_2022-09-15_detection_chi2.png', 'tmp_2611.25120/./cEri_2024-11-15_detection_chi2.png', 'tmp_2611.25120/./cEri_2024-12-18_detection_chi2.png', 'tmp_2611.25120/./cEri_2025-10-03_detection_chi2.png', 'tmp_2611.25120/./cEri-1p-g23h-ueva-mcmc-orbits.png', 'tmp_2611.25120/./cEri_1p_g23h_pm_figure.png', 'tmp_2611.25120/./cEri-relrv-fiducial-mcmc-orbits.png']
copying  tmp_2611.25120/./cEri_2021-10-28-irdis_fmmf_all_gravi_pointings.png to _build/html/
copying  tmp_2611.25120/./cEri_2022-01-23_detection_chi2.png to _build/html/
copying  tmp_2611.25120/./cEri_2022-09-15_detection_chi2.png to _build/html/
copying  tmp_2611.25120/./cEri_2024-11-15_detection_chi2.png to _build/html/
copying  tmp_2611.25120/./cEri_2024-12-18_detection_chi2.png to _build/html/
copying  tmp_2611.25120/./cEri_2025-10-03_detection_chi2.png to _build/html/
copy

## Display the papers

Not necessary but allows for a quick check.

In [9]:
[display(Markdown(k[1])) for k in documents];

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand\aastex{AAS\TeX}$
$\newcommand\latex{La\TeX}$
$\newcommand{\um}{\text{ \textmu m}\xspace}$
$\newcommand{\Mjup}{M_{\rm{Jup}}\xspace}$
$\newcommand{\Rjup}{R_{\rm{Jup}}\xspace}$
$\newcommand{\todo}[1]{\textcolor{red}{[#1]}}$</div>



<div id="title">

# On the High Eccentricity Orbit of 51 Eri b and Constraints on Interactions with a Second Companion

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.25120-b31b1b.svg)](https://arxiv.org/abs/2611.25120)<mark>Appeared on: 2026-11-20</mark> -  _Re-submitted to the Astronomical Journal_

</div>
<div id="authors">

W. O. Balmer, et al. -- incl., <mark>G. Chauvin</mark>, <mark>T. Henning</mark>, <mark>L. Kreidberg</mark>, <mark>M. Samland</mark>

</div>
<div id="abstract">

**Abstract:** The eccentricity of an exoplanet's orbit is a key observational quantity that can be used to infer its dynamical past and predict its future orbital evolution. Classically, planets are expected to emerge from their protoplanetary cradles with roughly circular eccentricities; non-circular eccentricites arise from subsequent dynamical processing in the absence of the gas disk. We present an updated orbit for the directly imaged exoplanet 51 Eridani b, based on 5 years of 0.2 mas precision optical interferometry from the VLTI/GRAVITY(+) instrument. This planet is young ( ${\sim}20 \mathrm{Myr}$ old), temperate ( $T_{\rm eff}\sim650 \mathrm{K}$ ), and on a wide orbit ( $a{=}9.36 \mathrm{au}$ ) with an uncertain mass ( $M_{\rm b}\leq12 M_{\rm J}$ ).We measured the planet's motion to an exceptionally high precision, confirming that its orbit is very eccentric ( $e_b=0.574\pm0.003$ ). We observe significant residuals to the absolute astrometry of the host star based on a single planet orbit that could be attributable to reflex motion from a third body in the system. We construct a two companion orbit model that could explain the apparent reflex motion of 51 Eri A: this model predicts that the putative second companion has $M_c\in[1.0, 2900]_{95\%} M_{\rm Jup}$ and $a_c=[0.3, 3.1]_{95\%} \mathrm{au}$ . In this framework, a dynamical interaction between b and "c" could have kicked the exterior, lower mass b out onto its present, eccentric orbit. This could either represent an epoch of planet-planet scattering, or circum-binary formation then scattering, depending on the mass and orbital separation of the candidate companion and the dynamical stability of the system. If this signal is astrophysical, and not due to a systematic error, this candidate should be confirmable with Gaia DR4 epoch astrometry in the near future.

</div>

<div id="div_fig1">

<img src="tmp_2611.25120/./cEri_2021-10-28-irdis_fmmf_all_gravi_pointings.png" alt="Fig6.1" width="16%"/><img src="tmp_2611.25120/./cEri_2022-01-23_detection_chi2.png" alt="Fig6.2" width="16%"/><img src="tmp_2611.25120/./cEri_2022-09-15_detection_chi2.png" alt="Fig6.3" width="16%"/><img src="tmp_2611.25120/./cEri_2024-11-15_detection_chi2.png" alt="Fig6.4" width="16%"/><img src="tmp_2611.25120/./cEri_2024-12-18_detection_chi2.png" alt="Fig6.5" width="16%"/><img src="tmp_2611.25120/./cEri_2025-10-03_detection_chi2.png" alt="Fig6.6" width="16%"/>

**Figure 6. -** Detections of 51 Eri b. (a) SPHERE coronagraphic imaging. The planet is detected at $5 \sigma$ in the _K1_-band. Subsequent GRAVITY pointing are overlaid in light blue, illustrating the instrument field of view relative to the planetary system scene. (b-f) VLTI/GRAVITY(+) interferometry. $\chi^2_{\rm no  planet}-\chi^2_{\rm planet}$ represents the likelihood the data can be described by a combination of an incoherent polynomial term capturing the stellar light leakage and instrumental/telluric effects and a coherent point source with a given on-sky position and the spectrum of a T-dwarf. This model was computed across the science fiber field of view (${\sim}60 \mathrm{mas}$ in $\Delta$RA and $\Delta$Dec in milliarcseconds).SPHERE, 2021-10-28GRAVITY, 2022-01-23GRAVITY, 2022-09-15GRAVITY+, 2024-11-15GRAVITY+, 2024-12-18GRAVITY+, 2025-10-03 (*fig:detections*)

</div>
<div id="div_fig2">

<img src="tmp_2611.25120/./cEri-1p-g23h-ueva-mcmc-orbits.png" alt="Fig7.1" width="50%"/><img src="tmp_2611.25120/./cEri_1p_g23h_pm_figure.png" alt="Fig7.2" width="50%"/>

**Figure 7. -** The results of the orbit fit including the G23H absolute astrometry. (a) _Top and middle panels_: same as in Figure \ref{fig:hgca_1p_orbit}. _Bottom_: Z-scores (normalized residuals, equivalent to $\sigma$ standard deviations) for each absolute astrometric measurement from the G23H catalog. A black horizontal line denotes a z-score of 0, and gray horizontal lines mark z-scores of 1. A larger z-score indicates that the measurement is in greater tension with the model. (b) _Top to bottom_: The G23H catalog values for the proper motion of the system (colored errorbars, which include the correlations between R.A. and Decl.), compared against the instantaneous predictions from the best fit linear motion and one planet orbit (colored squares, shaded by the mass of the planet). This visualization provides a more detailed view of the data and goodness of fit metric in the lower half of (a). (*fig:g23h_1p_orbit*)

</div>
<div id="div_fig3">

<img src="tmp_2611.25120/./cEri-relrv-fiducial-mcmc-orbits.png" alt="Fig2" width="100%"/>

**Figure 2. -** The results of the fiducial orbit fit including all relative astrometry and relative radial velocities. _Top_: Visual orbit of the planet; measurements from GRAVITY are plotted as red scatter points, from coronagraphic telescopes (Gemini, Keck, VLT, JWST) as blue scatter points. 250 random orbits from the posterior distribution are color-coded according to their mean anomaly $M$, the fraction of the orbital period that the planet has completed since its closest approach to the star. _Middle_: Residuals from the maximum a-posteriori orbit in R.A. and Decl. versus time, for the same orbits and observations. _Bottom_: Relative radial velocity versus time; HiRISE observations are plotted as white scatter points. (*fig:fiducial_orbit*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.25120"></div>

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\vdag}{(v)^\dagger}$
$\newcommand\aastex{AAS\TeX}$
$\newcommand\latex{La\TeX}$
$\newcommand{\jointage}{165^{+63}_{-34}}$
$\newcommand{\red}{\textcolor{red}}$
$\newcommand{\magenta}[1]{{\textcolor{magenta}#1}}$
$\newcommand{\blue}[1]{{\textcolor{blue}#1}}$
$\newcommand{\orange}[1]{{\textcolor{orange}#1}}$</div>



<div id="title">

# The SPACE Program IV: Atmospheric Features on the Cold Transiting Sub-Neptune TOI-6478 b

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.25230-b31b1b.svg)](https://arxiv.org/abs/2611.25230)<mark>Appeared on: 2026-11-20</mark> -  _16 pages, 9 Figure, 4 Tables, Submitted to AJ_

</div>
<div id="authors">

A. Triantafillides, et al. -- incl., <mark>A. Kahle</mark>, <mark>L. Kreidberg</mark>

</div>
<div id="abstract">

**Abstract:** Among the current population of exoplanets there are very few sub-Neptunes that have their atmospheric composition identified. This is primarily due to sub-Neptunes having high metallicity, clouds, and/or hazes obscuring absorption features. This often produces a spectrum that presents itself as a featureless spectrum especially in the near-IR. However, there are tentative trends that suggest sub-Neptunes with the lowest equilibrium temperatures ( $\mathrm{T_{eq} < 400\:K}$ ) have detectable features, perhaps due to inefficient haze production or cloud sedimentation. TOI-6478 b is a low-density sub-Neptune exoplanet ( $\rm <\mathrm{9.9\:M_{\oplus}}$ and $\rm \sim 4.5R_{\oplus}$ ) with an equilibrium temperature $\mathrm{T_{eq} = 220 \:K}$ assuming zero albedo. In this study, we present the highest S/N spectral feature ( $\sim$ 6) of any sub-Neptune detected using HST/WFC3. We detect this spectral feature using a robust analysis that incorporates the Transit Light Source Effect (TLSE) to rule out false detections due to stellar activity. With the narrow bandpass of HST we were unable to distinguish whether the spectral feature at 1.4 $\mu$ m is from $H_2$ O or $CH_4$ absorption. Meanwhile, self-consistent atmospheric models provide physical motivation for $CH_4$ to be the dominant absorber given that $H_2$ O should condense or cold trap at higher pressures. Through our analysis we determine that HST/WFC3 alone cannot distinguish between major absorbers. However, a broader wavelength coverage by JWST will be able to measure the atmospheric composition in detail, due to the high S/N features for this target.

</div>

<div id="div_fig1">

<img src="tmp_2611.25230/./forward_model_toi6478b.png" alt="Fig8" width="100%"/>

**Figure 8. -** **Panel a:** HST/WFC3 data plotted with the simulated transmission spectrum from the self-consistent forward model. The model parameters for these spectra are $5\rm \:M_{\oplus}$, $\rm M/H = +0.5$, and $\rm C/O = 1.0$, where the $f_{\rm sed}=5$ model is plotted in magenta and the cloud-free model plotted in purple. The points in black are the data from HST/WFC3 with 1 sigma error bars. **Panel b:** PT profiles from the self consistent forward models. Both lines use the same model parameters as in panel a. The gray dashed lines represent condensation curves for various condensables in the atmosphere. **Panel c:** Volume mixing ratio of $H_2$O from the self consistent forward models using the same models parameters as panels a and b. (*fig:forward_model*)

</div>
<div id="div_fig2">

<img src="tmp_2611.25230/./toi-6478_all_stis.png" alt="Fig1" width="100%"/>

**Figure 1. -** STIS spectroscopy of TOI-6478. Top: The G430L spectrum. Middle: Section of the stacked G230L flt images, showing the $\ion${Mg}{2} 2800Å lines but no other features. Bottom: Comparison of G230L spectra around $\ion${Mg}{2} extracted from just the first observation and from the stacked flt image.  (*fig:stis*)

</div>
<div id="div_fig3">

<img src="tmp_2611.25230/./gauss_v_line_TOI6478b.png" alt="Fig2" width="100%"/>

**Figure 2. -** Top panel: Primary analysis in black with one sigma error bars and it's corresponding Gaussian model in red and flat line in blue. Bottom panel: same as top panel but for the secondary analysis. We find that the Gaussian model was preferred for both analyses with a $\Delta\ln{\mathrm{B}} = 22.8$ and $\Delta\ln{\mathrm{B}} = 17.28$ for the primary and secondary analyses respectively. We also find that they are consistent with one another by calculating a Bayes factor between the primary and secondary analyses of $\Delta\ln{\mathrm{B}} = 3.51$. (*fig:reduction comparison*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.25230"></div>

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\ph}[1]{{\color{Magenta}{#1}}}$
$\newcommand{\arraystretch}{1.25}$
$\newcommand{\arraystretch}{1.25}$
$\newcommand{\arraystretch}{1.25}$
$\newcommand{\arraystretch}{1.25}$
$\newcommand{\arraystretch}{1.25}$
$\newcommand{\arraystretch}{1.25}$
$\newcommand{\arraystretch}{1.25}$
$\newcommand{\arraystretch}{1.25}$
$\newcommand{\arraystretch}{1.25}$
$\newcommand{\arraystretch}{1.25}$
$\newcommand{\arraystretch}{1.25}$
$\newcommand{\arraystretch}{1.25}$
$\newcommand{\arraystretch}{1.25}$
$\newcommand{\arraystretch}{1.25}$
$\newcommand{\arraystretch}{1.25}$
$\newcommand{\arraystretch}{1.25}$
$\newcommand{\arraystretch}{1.25}$
$\newcommand{\arraystretch}{1.25}$
$\newcommand{\thebibliography}{\DeclareRobustCommand{\VAN}[3]{##3}\VANthebibliography}$</div>



<div id="title">

# Square Kilometre Array Science Data Challenge 3b: EoR inference

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.25823-b31b1b.svg)](https://arxiv.org/abs/2611.25823)<mark>Appeared on: 2026-11-20</mark> -  _32 pages, 13 figures, submitted to MNRAS_

</div>
<div id="authors">

O. Bait, et al. -- incl., <mark>B. Maity</mark>

</div>
<div id="abstract">

**Abstract:** We present and analyse the results of Science Data Challenge 3b (SDC3b; $\url{https://sdc3.skao.int/challenges/inference}$ ), an Epoch of Reionization (EoR) inference exercise organised by the Square Kilometre Array Observatory (SKAO) using simulated SKA data. The nine-month challenge, running from October 2024 to May 2025, represents the first community-wide comparison of EoR inference pipelines on common simulated datasets. We consider multiple datasets designed to assess performance on both foreground-free and foreground-contaminated cylindrical power spectra and imaging products, adopting different reionization scenarios and EoR simulation models. We include results from 16 teams, which employ a range of methodologies to infer the posterior distribution of the hydrogen neutral fraction across three redshift intervals spanning $z=8.41$ to $z=6.25$ . The results show significant inter-team variability of the astrophysical constraints even in the simpler case where only instrumental noise is added to the cosmological signal. This underscores the need for further investigation into the relative contributions of signal modelling, the treatment of instrumental effects, and inference methods to the robustness of the resulting conclusions. Although individual methods can produce biased parameter estimates, the combined average of their estimates remains close to the input values, suggesting that independent approaches can offset method-specific biases.

</div>

<div id="div_fig1">

<img src="tmp_2611.25823/./figures/Fig4.png" alt="Fig8" width="100%"/>

**Figure 8. -** Boxplot (see Sec. \ref{sec:forest}) for the PS1 results. The boxes and bars represent $\Delta_{68}$ and $\Delta_{95}$, respectively. The symbols are the marginal median $\bar{x}$, the vertical dashed line is the central input value $x'$ and the shaded area its evolution range $[a,b]$ across the light-cone.
 (*fig:forest1*)

</div>
<div id="div_fig2">

<img src="tmp_2611.25823/./figures/Fig1.png" alt="Fig1" width="100%"/>

**Figure 1. -** Neutral fraction as a function of redshift for the three datasets. The frequency ranges delivered are also displayed.  (*fig:modelcomp*)

</div>
<div id="div_fig3">

<img src="tmp_2611.25823/./figures/Fig5.png" alt="Fig4" width="100%"/>

**Figure 4. -** Comparison of the SDC3b score $S_\mathrm{SDC3b}$ and the Mahalanobis distance $D_\mathrm{M}$ for the PS1 results. The markers are defined in Fig. \ref{fig:legend}. The gray horizontal band represents the $D_\mathrm{M}<1$ area, outside of which solutions are overconfident. The dashed vertical line is the maximum $S_\mathrm{SDC3b}$ score.  (*fig:scores1*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.25823"></div>

# Create HTML index

In [10]:
from datetime import datetime, timedelta, timezone
from glob import glob
import os

files = glob('_build/html/*.md')
days = 7
now = datetime.today()
res = []
for fk in files:
    stat_result = os.stat(fk).st_ctime
    modified = datetime.fromtimestamp(stat_result, tz=timezone.utc).replace(tzinfo=None)
    delta = now.today() - modified
    if delta <= timedelta(days=days):
        res.append((delta.seconds, fk))
res = [k[1] for k in reversed(sorted(res, key=lambda x:x[1]))]
npub = len(res)
print(len(res), f" publications files modified in the last {days:d} days.")
# [ print('\t', k) for k in res ];

158  publications files modified in the last 7 days.


In [11]:
import datetime
from glob import glob

def get_last_n_days(lst, days=1):
    """ Get the documents from the last n days """
    sorted_lst = sorted(lst, key=lambda x: x[1], reverse=True)
    for fname, date in sorted_lst:
        if date >= str(datetime.date.today() - datetime.timedelta(days=days)):
            yield fname

def extract_appearance_dates(lst_file):
    dates = []

    def get_date(line):
        return line\
            .split('Appeared on:')[-1]\
            .split('</mark>')[0].strip()

    for fname in lst:
        with open(fname, 'r') as f:
            found_date = False
            for line in f:
                if not found_date:
                    if "Appeared on" in line:
                        found_date = True
                        dates.append((fname, get_date(line)))
                else:
                    break
    return dates

from glob import glob
lst = glob('_build/html/*md')
days = 7
dates = extract_appearance_dates(lst)
res = list(get_last_n_days(dates, days))
npub = len(res)
print(len(res), f" publications in the last {days:d} days.")

23  publications in the last 7 days.


In [12]:
def create_carousel(npub=4):
    """ Generate the HTML code for a carousel with `npub` slides """
    carousel = ["""  <div class="carousel" """,
                """       data-flickity='{ "autoPlay": 10000, "adaptiveHeight": true, "resize": true, "wrapAround": true, "pauseAutoPlayOnHover": true, "groupCells": 1 }' id="asyncTypeset">"""
                ]
    
    item_str = """    <div class="carousel-cell"> <div id="slide{k}" class="md_view">Content {k}</div> </div>"""
    for k in range(1, npub + 1):
        carousel.append(item_str.format(k=k))
    carousel.append("  </div>")
    return '\n'.join(carousel)

def create_grid(npub=4):
    """ Generate the HTML code for a flat grid with `npub` slides """
    grid = ["""  <div class="grid"> """,
                ]
    
    item_str = """    <div class="grid-item"> <div id="slide{k}" class="md_view">Content {k}</div> </div>"""
    for k in range(1, npub + 1):
        grid.append(item_str.format(k=k))
    grid.append("  </div>")
    return '\n'.join(grid)

In [13]:
carousel = create_carousel(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("daily_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- carousel:s --%}", carousel)\
               .replace("{%-- suptitle:s --%}",  "7-day archives" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
with open("_build/html/index_7days.html", 'w') as fout:
    fout.write(page)

In [14]:
# redo for today
days = 1
res = list(get_last_n_days(dates, days))
npub = len(res)
print(len(res), f" publications in the last day.")

carousel = create_carousel(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("daily_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- carousel:s --%}", carousel)\
               .replace("{%-- suptitle:s --%}",  "Daily" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
# print(carousel, docs, slides)
# print(page)
with open("_build/html/index_daily.html", 'w') as fout:
    fout.write(page)

6  publications in the last day.


In [15]:
# Create the flat grid of the last N papers (fixed number regardless of dates)
from itertools import islice 

npub = 6
res = [k[0] for k in (islice(reversed(sorted(dates, key=lambda x: x[1])), 6))]
print(len(res), f" {npub} publications selected.")

grid = create_grid(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("grid_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- grid-content:s --%}", grid)\
               .replace("{%-- suptitle:s --%}",  f"Last {npub:,d} papers" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
# print(grid, docs, slides)
# print(page)
with open("_build/html/index_npub_grid.html", 'w') as fout:
    fout.write(page)

6  6 publications selected.
